# 第10回：モデル対決

**今日の問い：複雑なモデルは本当にいつも優れているか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 同じ分割・指標で複数モデルを比較し、性能・速度・安定性を並べる
- 反復交差検証と対応のある検定で、差が偶然でないかを確かめる
- 投票・スタッキングで複数モデルを束ね、単体との差を評価する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 反復交差検証：分割の乱数を変えて繰り返す評価
- 対応のある検定：同じ分割上の差を比べる統計的検定
- 投票分類器：複数モデルの多数決や平均確率で決めるモデル
- スタッキング：モデルの予測を入力に上位モデルで統合する方法
- 勾配ブースティング：前の誤りを順に補正する木の集合

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 「複雑なモデルほど強い」は本当か

新しいモデルを次々試したくなりますが、この回で確かめるのは**「複雑さは必ずしも勝たない」**という
実感です。大事なのは勝ち負けそのものより、**フェアな比べ方**を身につけること。フェアな比較には
3つの「同じ」が要ります：**同じ分割・同じ指標・同じ前処理**。

まず、単純〜複雑まで5つのモデルを1つの辞書にまとめます。前処理が要るモデルは`make_pipeline`で
前処理込みにしてあるので、どれも同じ`X_train`をそのまま渡せます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=200, random_state=42),
}


## TRY：同じ土俵で、F1と学習時間を並べる

5モデルを同じデータで学習し、**検証F1**と**学習にかかった秒数**を並べます。性能だけでなく
**コスト（時間）**も一緒に見るのが実務的な比較です。


In [ ]:
rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    rows.append({"モデル": name, "検証F1": f1_score(y_valid, model.predict(X_valid)), "学習秒": elapsed})
pd.DataFrame(rows).sort_values("検証F1", ascending=False).round({"検証F1": 3, "学習秒": 4})


### 出力の読み方

- **Dummyが最下位**なのは当然。他がDummyをどれだけ引き離すかが価値です。
- **最も複雑なモデルが1位とは限りません**。線形モデルが健闘したり、木モデルと僅差だったりします。差が小さいなら、**速くて説明しやすいモデル**を選ぶ理由になります。
- ただし、これは**1回の分割の結果**。順位が分割運で入れ替わるかもしれません。次で交差検証により安定性を確かめます。


## CORE深掘り：交差検証で「安定して強いか」を見る

1回の勝敗は運に左右されます。交差検証で**平均F1・ばらつき(標準偏差)・最低F1**を出し、
「平均が高い」だけでなく「**転んでも大崩れしない**（最低F1が高い）」モデルを評価します。


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=42)
stability = []
for name, estimator in models.items():
    scores = cross_val_score(estimator, df[features], df["active"], cv=cv, scoring="f1")
    stability.append({"モデル": name, "F1平均": scores.mean(), "F1標準偏差": scores.std(), "最低F1": scores.min()})
pd.DataFrame(stability).sort_values("F1平均", ascending=False).round(3)


### 出力の読み方

- **F1平均**で総合力、**F1標準偏差**で安定度、**最低F1**で最悪ケースを見ます。
- 平均が僅差なら、**標準偏差が小さい方**が実務では安心。平均1位でも最低F1が極端に低いモデルは、条件次第で大外しする危険があります。
- 1回の分割（前セル）と順位が入れ替わることもあります。だから**単発の勝敗で決めない**——これがこの回の教訓です。


## 5人の担当

Dummy / Logistic / Tree / Random Forest / Gradient Boosting を1人ずつ担当し、
**スコア・学習時間・説明しやすさ・安定性**を1行で共有します。「どれが最強か」ではなく
「**この用途にはどれが妥当か**」を言葉にするのがゴールです。


## DEEP DIVE：その差は「偶然」か、そして束ねる価値

上位2モデルのF1差が0.01だったとして、それは本物の差でしょうか、それとも分割運でしょうか。
ここでは**反復交差検証＋統計的検定**で差の確からしさを測り、次に複数モデルを**束ねる**価値を見ます。


### 反復CV＋Wilcoxon検定：差は偶然でないか

分割の乱数を変えて交差検証を何度も繰り返し（反復CV）、上位2モデルのスコア列を**対応のある検定
（Wilcoxon）**で比べます。p値が小さいほど「差は偶然では説明しにくい」と読めます。


In [ ]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import wilcoxon

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)
dist = {name: cross_val_score(est, df[features], df["active"], cv=rcv, scoring="f1") for name, est in models.items()}
summary = pd.DataFrame({name: {"F1平均": s.mean(), "F1_SD": s.std()} for name, s in dist.items()}).T.sort_values("F1平均", ascending=False)
display(summary.round(3))
top2 = summary.index[:2].tolist()
stat, p = wilcoxon(dist[top2[0]], dist[top2[1]])
print(f"{top2[0]} vs {top2[1]} のWilcoxon検定 p={p:.3f}（小さいほど差が偶然でない）")


### 出力の読み方

- **p値が0.05より大きい**なら、上位2モデルの差は「偶然の範囲」かもしれず、**わざわざ複雑な方を選ぶ理由は弱い**。
- p値が小さくても、差の**大きさ**（実務的な意味があるか）は別問題。「統計的に有意」と「実務的に重要」は違う、という感覚を持ちます。


### 投票・スタッキングで束ねる

間違え方の違うモデルを組み合わせると、単体より安定することがあります。**Voting**は予測確率の平均、
**Stacking**は各モデルの予測を入力に上位モデルで統合します。単体最良と比べます。


In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [(name, est) for name, est in models.items() if name != "Dummy"]
ensembles = {
    "Voting(soft)": VotingClassifier(estimators, voting="soft"),
    "Stacking": StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=1000), cv=5),
}
for name, est in ensembles.items():
    scores = cross_val_score(est, df[features], df["active"], cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1")
    print(f"{name:14s} F1平均={scores.mean():.3f} ± {scores.std():.3f}")
print("最良単体:", summary.index[0], "F1平均=", round(summary.iloc[0, 0], 3))


### 出力の読み方

束ねたモデルが最良単体を**明確に上回るとは限りません**。束ねる価値が出るのは、元のモデルたちが
「互いに違う間違え方」をするとき。差がわずかなら、運用の手間を考えて単体を選ぶのも正解です。


### 任意：勾配ブースティング専用ライブラリ

XGBoostが入っていれば試します（`uv sync --extra advanced`）。無い環境では自動でメッセージを出して
スキップし、sklearnの`HistGradientBoosting`で代用できます。


In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42, eval_metric="logloss")
    scores = cross_val_score(xgb, df[features].fillna(df[features].median()), df["active"], cv=5, scoring="f1")
    print("XGBoost F1平均:", round(scores.mean(), 3))
except ImportError:
    print("XGBoostは任意です（uv sync --extra advanced）。HistGradientBoostingで代用できます。")


### 出力の読み方

XGBoostのF1が、既に見たGradient Boostingと**近い値**になるはずです。「専用ライブラリ＝必ず勝つ」では
ありません。ライブラリの新しさより、**フェアな比較の枠組み**の方がずっと大事、という締めくくりです。


## よくある誤り

- 異なる分割で比較する
- モデルごとに異なる指標を報告する
- 最も高い1回のスコアだけを採用する

## SELF-STUDY（任意・30〜60分）

- RepeatedStratifiedKFoldでF1分布を作り、上位2モデルをWilcoxon検定で比べる
- StackingClassifierと最良単体のF1・学習時間を比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 公平な比較に固定すべきものは何か
2. 対応のある検定が必要な理由は何か
3. スタッキングが効きやすいのはどんなときか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
